<a href="https://colab.research.google.com/github/jdavenport8990/Python-Spring-26/blob/main/Week15/AI_Assignment2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import json
import os
from datetime import datetime, timedelta

class Plant:
    def __init__(self, name, water_days, last_watered=None):
        self.name = str(name).strip()
        # Edge Case: Ensure water_days is always a positive integer
        try:
            self.water_days = max(1, int(water_days))
        except (ValueError, TypeError):
            self.water_days = 1 # Default fallback

        self.last_watered = last_watered if last_watered else datetime.now().strftime("%Y-%m-%d")

    def days_until_thirsty(self):
        try:
            last_dt = datetime.strptime(self.last_watered, "%Y-%m-%d")
            next_water_dt = last_dt + timedelta(days=self.water_days)
            # Compare to 'now' without hours/minutes for date-only accuracy
            today = datetime.now().replace(hour=0, minute=0, second=0, microsecond=0)
            delta = next_water_dt - today
            return delta.days
        except ValueError:
            return 0 # Fallback if date string is corrupted

    def to_dict(self):
        return {"name": self.name, "water_days": self.water_days, "last_watered": self.last_watered}

class GardenManager:
    def __init__(self):
        self.data_file = "garden_data.json"
        self.plants = self.load_data()

    def load_data(self):
        """Handle corrupted or missing files."""
        if not os.path.exists(self.data_file):
            return []
        try:
            with open(self.data_file, "r") as f:
                data = json.load(f)
                return [Plant(**p) for p in data]
        except (json.JSONDecodeError, TypeError, KeyError):
            print("⚠️ Warning: Data file corrupted. Starting with an empty garden.")
            return []

    def save_data(self):
        try:
            with open(self.data_file, "w") as f:
                json.dump([p.to_dict() for p in self.plants], f, indent=4)
        except IOError as e:
            print(f"❌ Error saving data: {e}")

    def get_valid_int(self, prompt, min_val=1):
        """Reusable helper to prevent non-integer crashes."""
        while True:
            try:
                val = int(input(prompt))
                if val < min_val:
                    print(f"Please enter a number {min_val} or higher.")
                    continue
                return val
            except ValueError:
                print("❌ Invalid input. Please enter a whole number.")

    def add_plant(self):
        name = input("\nPlant name: ").strip()
        if not name:
            print("❌ Name cannot be empty.")
            return
        days = self.get_valid_int("Watering interval (days): ")
        self.plants.append(Plant(name, days))
        self.save_data()
        print(f"✅ {name} added!")

    def check_status(self):
        if not self.plants:
            print("\nYour garden is currently empty.")
            return

        print("\n--- 🌿 Garden Status ---")
        # Logic: Sort by urgency (overdue plants at the top)
        sorted_plants = sorted(self.plants, key=lambda x: x.days_until_thirsty())

        for i, p in enumerate(sorted_plants, 1):
            days = p.days_until_thirsty()
            if days < 0:
                msg = f"🚨 OVERDUE by {abs(days)} days"
            elif days == 0:
                msg = "💧 Due TODAY"
            else:
                msg = f"✅ Healthy (Due in {days} days)"
            print(f"{i}. {p.name:<15} | {msg}")

    def water_plant(self):
        if not self.plants: return
        self.check_status()
        choice = self.get_valid_int("\nEnter the number of the plant you watered: ")

        if 1 <= choice <= len(self.plants):
            # Account for sorting in display vs index in list
            sorted_plants = sorted(self.plants, key=lambda x: x.days_until_thirsty())
            target_plant = sorted_plants[choice - 1]
            target_plant.last_watered = datetime.now().strftime("%Y-%m-%d")
            self.save_data()
            print(f"💦 {target_plant.name} updated!")
        else:
            print("❌ Invalid plant number.")

def main():
    garden = GardenManager()
    menu = {
        "1": garden.add_plant,
        "2": garden.check_status,
        "3": garden.water_plant,
        "4": exit
    }

    while True:
        print("\n--- 🌻 PRO GARDEN TRACKER ---")
        print("1. Add Plant\n2. Check Status\n3. Water Plant\n4. Exit")
        choice = input("Select: ").strip()

        if choice in menu:
            if choice == "4":
                print("Goodbye!")
                break
            menu[choice]()
        else:
            print("⚠️ Invalid selection. Please choose 1-4.")

if __name__ == "__main__":
    main()


--- 🌻 PRO GARDEN TRACKER ---
1. Add Plant
2. Check Status
3. Water Plant
4. Exit
Select: 2

--- 🌿 Garden Status ---
1. Watermelon      | ✅ Healthy (Due in 1 days)
2. Zucchini        | ✅ Healthy (Due in 1 days)
3. Zucchini        | ✅ Healthy (Due in 2 days)
4. Cucumber        | ✅ Healthy (Due in 2 days)
5. Green Beans     | ✅ Healthy (Due in 3 days)

--- 🌻 PRO GARDEN TRACKER ---
1. Add Plant
2. Check Status
3. Water Plant
4. Exit
Select: 1

Plant name: Corn
Watering interval (days): 10
✅ Corn added!

--- 🌻 PRO GARDEN TRACKER ---
1. Add Plant
2. Check Status
3. Water Plant
4. Exit
Select: 3

--- 🌿 Garden Status ---
1. Watermelon      | ✅ Healthy (Due in 1 days)
2. Zucchini        | ✅ Healthy (Due in 1 days)
3. Zucchini        | ✅ Healthy (Due in 2 days)
4. Cucumber        | ✅ Healthy (Due in 2 days)
5. Green Beans     | ✅ Healthy (Due in 3 days)
6. Corn            | ✅ Healthy (Due in 10 days)

Enter the number of the plant you watered: 5
💦 Green Beans updated!

--- 🌻 PRO GARDEN TRACKER 